# Retailrocket EDA — W1

**목적**: S1 임계값(`tab_hidden_seconds`) 및 부스터 가중치 산출 근거 확보  
**데이터**: Retailrocket Recommender System Dataset (Kaggle)  
**결과물**: `outputs/` 폴더 PNG 4종 + `thresholds.yml` PR #1 근거

---

## 결론 요약 (분석 후 이 셀 업데이트)

| 항목 | 값 | 비고 |
|---|---|---|
| 전체 이벤트 수 | - | |
| 고유 사용자 수 | - | |
| 전체 세션 수 | - | |
| 카트 추가 세션 비율 | - | |
| 카트→30초 내 전환율 | - | ← S1 임계값 근거 |
| 카트→변곡점 N초 | - | ← 권장 임계값 |
| 5분+ 세션 구매 lift | - | ← session_length_5min 가중치 근거 |

### thresholds.yml 권장 갱신값
- `S1.tab_hidden_seconds`: 30 → **___** (분석 후 채워넣기)
- `booster_weights.session_length_5min`: 0.1 → **___**
- `booster_weights.hidden_repeated`: 0.1 → **___**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (Windows)
plt.rcParams.update({
    'font.family': 'Malgun Gothic',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False
})

print('라이브러리 로드 완료')

## 1.1 데이터 로딩 및 기초 통계

In [ ]:
events = pd.read_csv('data/events.csv')

print('=== 데이터 형태 ===')
print(f'Shape: {events.shape}')
print(f'Columns: {events.columns.tolist()}')
print()
print(events.dtypes)
print()
print(events.head())

In [ ]:
print(f'전체 이벤트: {len(events):,}')
print(f'고유 사용자: {events["visitorid"].nunique():,}')
print(f'고유 아이템: {events["itemid"].nunique():,}')
print()
print('이벤트 타입별 수:')
print(events['event'].value_counts())
print()

events['datetime'] = pd.to_datetime(events['timestamp'], unit='ms')
print(f'기간: {events["datetime"].min()} ~ {events["datetime"].max()}')
print(f'결측치:\n{events.isnull().sum()}')

In [ ]:
# 이벤트 타입 분포 저장
fig, ax = plt.subplots(figsize=(8, 5))
event_counts = events['event'].value_counts()
bars = ax.bar(event_counts.index, event_counts.values, color=['#3498db', '#f39c12', '#27ae60'])
for bar, val in zip(bars, event_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:,}\n({val/len(events)*100:.1f}%)',
            ha='center', va='bottom', fontsize=11)
ax.set_title('Retailrocket 이벤트 타입 분포')
ax.set_ylabel('이벤트 수')
plt.tight_layout()
plt.savefig('outputs/event_type_counts.png', bbox_inches='tight')
plt.show()
print('저장: outputs/event_type_counts.png')

## 1.2 세션 분리 (30분 inactivity 기준)

In [ ]:
SESSION_GAP_MS = 30 * 60 * 1000  # 30분

events = events.sort_values(['visitorid', 'timestamp']).reset_index(drop=True)
events['time_diff'] = events.groupby('visitorid')['timestamp'].diff()
events['new_session'] = (events['time_diff'] > SESSION_GAP_MS) | events['time_diff'].isna()
events['session_num'] = events.groupby('visitorid')['new_session'].cumsum()
events['session_id'] = events['visitorid'].astype(str) + '_' + events['session_num'].astype(str)

print(f'세션 ID 부여 완료')
print(f'고유 세션 수: {events["session_id"].nunique():,}')

In [ ]:
session_stats = events.groupby('session_id').agg(
    visitor_id=('visitorid', 'first'),
    start_ts=('timestamp', 'min'),
    end_ts=('timestamp', 'max'),
    event_count=('event', 'count'),
    has_view=('event', lambda x: (x == 'view').any()),
    has_addtocart=('event', lambda x: (x == 'addtocart').any()),
    has_transaction=('event', lambda x: (x == 'transaction').any())
).reset_index()
session_stats['duration_sec'] = (session_stats['end_ts'] - session_stats['start_ts']) / 1000

print(f'전체 세션: {len(session_stats):,}')
print(f'카트 추가 세션: {session_stats["has_addtocart"].sum():,} ({session_stats["has_addtocart"].mean()*100:.1f}%)')
print(f'구매 세션: {session_stats["has_transaction"].sum():,} ({session_stats["has_transaction"].mean()*100:.2f}%)')
print()
print('세션당 이벤트 수 통계:')
print(session_stats['event_count'].describe())

## 1.3 ★ 카트→이탈 시간 분포 (S1 임계값 근거)

In [ ]:
cart_events = events[events['event'] == 'addtocart'].copy()
cart_events = cart_events.rename(columns={'timestamp': 'cart_ts'})[['session_id', 'cart_ts']]
cart_sessions_set = set(cart_events['session_id'].unique())

events_after_cart = events.merge(cart_events, on='session_id')
events_after_cart = events_after_cart[events_after_cart['timestamp'] > events_after_cart['cart_ts']]
events_after_cart['time_since_cart_sec'] = (
    events_after_cart['timestamp'] - events_after_cart['cart_ts']
) / 1000

print(f'카트 이후 이벤트 수: {len(events_after_cart):,}')

In [ ]:
# 세밀한 bucket (5초 단위, 최대 10분)
fine_buckets = list(range(5, 601, 5))
fine_results = []

for bucket in fine_buckets:
    txns_within = events_after_cart[
        (events_after_cart['event'] == 'transaction') &
        (events_after_cart['time_since_cart_sec'] <= bucket)
    ]['session_id'].nunique()
    pct = txns_within / len(cart_sessions_set) * 100
    fine_results.append({'sec': bucket, 'pct': pct})

df_fine = pd.DataFrame(fine_results)
df_fine['delta'] = df_fine['pct'].diff()

# 변곡점: 5초당 증가율이 0.05% 미만이 되는 첫 지점
inflection_candidates = df_fine[df_fine['delta'] < 0.05]
if len(inflection_candidates) > 0:
    RECOMMENDED_N = int(inflection_candidates.iloc[0]['sec'])
else:
    RECOMMENDED_N = 30

pct_at_N = df_fine[df_fine['sec'] == RECOMMENDED_N]['pct'].values
pct_at_30 = df_fine[df_fine['sec'] == 30]['pct'].values

print(f'카트 후 30초 내 전환율: {pct_at_30[0]:.2f}%' if len(pct_at_30) else '30초 데이터 없음')
print(f'\n💡 권장 임계값 N = {RECOMMENDED_N}초')
print(f'   이 시점까지 누적 전환율: {pct_at_N[0]:.2f}%' if len(pct_at_N) else '')
print()
print('변곡점 후보 (5초당 전환율 증가 < 0.05%):')
print(inflection_candidates.head(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 왼쪽: 전체 곡선
axes[0].plot(df_fine['sec'], df_fine['pct'], linewidth=2, color='#2980b9')
axes[0].axvline(x=30, color='red', linestyle='--', alpha=0.7, label='30초 (현재 임계값)')
axes[0].axvline(x=RECOMMENDED_N, color='green', linestyle='--', alpha=0.7,
                label=f'{RECOMMENDED_N}초 (권장 임계값)')
axes[0].set_xlabel('카트 추가 후 경과 시간 (초)')
axes[0].set_ylabel('누적 전환율 (%)')
axes[0].set_title('카트 추가 후 시간별 누적 전환율')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 오른쪽: 5초당 증가율 (변곡점 확인)
axes[1].plot(df_fine['sec'][1:], df_fine['delta'][1:], linewidth=1.5, color='#e74c3c')
axes[1].axhline(y=0.05, color='gray', linestyle='--', alpha=0.7, label='임계 기울기 0.05%')
axes[1].axvline(x=RECOMMENDED_N, color='green', linestyle='--', alpha=0.7,
                label=f'변곡점 {RECOMMENDED_N}초')
axes[1].set_xlabel('카트 추가 후 경과 시간 (초)')
axes[1].set_ylabel('5초당 전환율 증가분 (%)')
axes[1].set_title('전환율 증가 속도 (기울기)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'S1 임계값 분석 — 권장값: {RECOMMENDED_N}초', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/cart_to_conversion_curve.png', bbox_inches='tight')
plt.show()
print('저장: outputs/cart_to_conversion_curve.png')

## 1.4 세션 길이 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(session_stats['duration_sec'].clip(upper=3600), bins=80, color='#3498db', alpha=0.7)
axes[0].axvline(x=300, color='red', linestyle='--', label='5분 기준')
axes[0].set_xlabel('세션 길이 (초)')
axes[0].set_ylabel('세션 수')
axes[0].set_title('전체 세션 길이 분포')
axes[0].legend()

cart_sessions_df = session_stats[session_stats['has_addtocart']]
axes[1].hist(cart_sessions_df['duration_sec'].clip(upper=3600), bins=80, color='#f39c12', alpha=0.7)
axes[1].axvline(x=300, color='red', linestyle='--', label='5분 기준')
axes[1].set_xlabel('세션 길이 (초)')
axes[1].set_ylabel('세션 수')
axes[1].set_title('카트 추가한 세션 길이 분포')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/session_duration_dist.png', bbox_inches='tight')
plt.show()
print('저장: outputs/session_duration_dist.png')

In [ ]:
long_sessions = session_stats[session_stats['duration_sec'] >= 300]
short_sessions = session_stats[session_stats['duration_sec'] < 300]

long_conv = long_sessions['has_transaction'].mean() * 100
short_conv = short_sessions['has_transaction'].mean() * 100
lift = long_conv / short_conv if short_conv > 0 else None

print(f'5분 이상 세션 구매율: {long_conv:.2f}%  (n={len(long_sessions):,})')
print(f'5분 미만 세션 구매율: {short_conv:.2f}%  (n={len(short_sessions):,})')
print(f'Lift: {lift:.2f}x')
print()
if lift and lift >= 1.5:
    print(f'✅ lift={lift:.2f}x ≥ 1.5x → session_length_5min 부스터 가중치 정당화됨')
    print(f'   권장 가중치 갱신: 0.1 → {min(0.1 * lift, 0.25):.2f}')
else:
    print(f'⚠️ lift={lift:.2f}x < 1.5x → 가중치 유지 또는 축소 검토 필요')

## 1.5 구매 깔때기 (Funnel)

In [ ]:
funnel_data = {
    'view': events[events['event'] == 'view']['session_id'].nunique(),
    'addtocart': events[events['event'] == 'addtocart']['session_id'].nunique(),
    'transaction': events[events['event'] == 'transaction']['session_id'].nunique()
}

fig, ax = plt.subplots(figsize=(10, 6))
labels = ['조회 (view)', '카트 추가 (addtocart)', '구매 (transaction)']
values = list(funnel_data.values())
colors = ['#3498db', '#f39c12', '#27ae60']

bars = ax.barh(labels, values, color=colors, height=0.5)
for bar, val in zip(bars, values):
    pct = val / values[0] * 100
    ax.text(val, bar.get_y() + bar.get_height()/2,
            f' {val:,}  ({pct:.1f}%)', va='center', fontsize=11)

ax.set_title('Retailrocket 구매 깔때기')
ax.set_xlabel('세션 수')
plt.tight_layout()
plt.savefig('outputs/funnel.png', bbox_inches='tight')
plt.show()
print('저장: outputs/funnel.png')

print(f'\nview → cart 전환율: {values[1]/values[0]*100:.2f}%')
print(f'cart → purchase 전환율: {values[2]/values[1]*100:.2f}%')

## 최종 결론 — thresholds.yml PR #1 근거

In [ ]:
print('=' * 60)
print('Retailrocket EDA 최종 결론')
print('=' * 60)
print(f'전체 이벤트: {len(events):,}')
print(f'고유 사용자: {events["visitorid"].nunique():,}')
print(f'전체 세션: {len(session_stats):,}')
print()
print(f'[S1 임계값]')
print(f'  현재 tab_hidden_seconds = 30초')
print(f'  권장 변경값: {RECOMMENDED_N}초 (전환율 곡선 변곡점)')
print()
print(f'[부스터 가중치]')
print(f'  session_length_5min: lift={lift:.2f}x → {"변경 권장" if lift and lift >= 1.5 else "유지 또는 축소"}')
print()
print('[한계]')
print('  - 클립보드 복사·탭 전환 신호 없음 (clipboard/broadcast_channel 도메인 직관으로 유지)')
print('  - 호텔이 아닌 일반 이커머스 데이터')